# Removing Camera Fog caused by Water-proof Pouch using Temporal Fusion Techniques

本 Notebook 實作報告中的方法：輸入一段因防水袋起霧而模糊的「移動影像」，整合多幀之間的資訊，還原出無霧的畫面。

**前提假設**
- 所有像素的運動一致（即相機運動），且相機不會往回移動
- 相機移動速度緩慢且穩定
- 剛體（例如岩石）主要位於畫面中央

**3-Stage Pipeline**

| 階段 | 內容 | 對應 Cell |
|---|---|---|
| 1. Global Warping | Focus Mask → SIFT → FLANN → 4-DOF Affine → Keyframe Anchoring / Middle Plane → 高斯軌跡平滑 | `align_video_keyframe_anchored` |
| 2. Temporal Fusion | 將所有幀投影到 Global Canvas，依 Fog Confidence 自適應篩選後取逐像素中位數 | `construct_global_image` |
| 3. Color Matching & Reprojection | 將 Global Canvas 反投影回每一幀，以 Lab 空間 Reinhard 色彩匹配 + EMA 對齊曝光後融合 | `construct_defog_vedio` |

**使用方式**：將起霧影片放在 `./data/foggy_vid/`（檔名 `1.mp4`、`2.mp4`…），依序執行所有 Cell，結果會輸出到 `./data/foggless_vid/`。Stage 2 需要 CUDA GPU 與 CuPy。

## 環境設定
匯入共用套件，關閉 NumPy 的 RuntimeWarning（`nanmean` 等函式遇到全 NaN 時的警告），並設定 matplotlib 顯示大小。

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import warnings

warnings.simplefilter("ignore", category=RuntimeWarning)

# matplotlib 顯示大小
plt.rcParams['figure.figsize'] = [18, 12]

## Fog Confidence Map
`get_confidence_map` 計算每個像素「受霧影響」的可能性（越高越可能有霧），由三個分數加權平均：

- **Dark Channel Prior**（`w_dcp=0.5`）：依 *Single Image Haze Removal Using Dark Channel Prior*，dark channel 值越低的像素越不可能受霧影響
- **Gradient magnitude**（`w_grad=0.3`）：受霧影響的區域較平滑，梯度較小，因此取 `1 - 正規化梯度`
- **Saturation**（`w_sat=0.2`）：受霧影響的區域飽和度較低，因此取 `1 - S`

加權後再乘上 `clip(2V, 0, 1)` 壓低暗部，並以 101×101 高斯模糊平滑。函式同時回傳以 `fog_threshold` 二值化、經形態學 close/open 處理的粗略遮罩。

此函式在 Stage 2（逐像素篩選）與 Stage 3（融合遮罩、色彩統計區域）中都會用到。

In [2]:
def get_confidence_map(image_bgr, patch_size=15, w_dcp=0.5, w_grad=0.3, w_sat=0.2, fog_threshold=0.6):
	gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
	hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV).astype(np.float32) / 255.0
	
	min_channel = np.min(image_bgr.astype(np.float32) / 255.0, axis=2)
	kernel_dcp = cv2.getStructuringElement(cv2.MORPH_RECT, (patch_size, patch_size))
	conf_dcp = cv2.erode(min_channel, kernel_dcp)
	
	sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
	sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
	grad_mag = np.sqrt(sobelx**2 + sobely**2)
	grad_norm = np.clip(grad_mag / (np.percentile(grad_mag, 95) + 1e-6), 0, 1)
	conf_grad = 1.0 - grad_norm
	
	sat = hsv[:, :, 1]
	conf_sat = 1.0 - sat
	
	total_w = w_dcp + w_grad + w_sat
	confidence_map = ((w_dcp/total_w) * conf_dcp + 
					(w_grad/total_w) * conf_grad + 
					(w_sat/total_w) * conf_sat)
	
	val = hsv[:, :, 2]
	confidence_map = confidence_map * np.clip(val * 2.0, 0, 1) 
	confidence_map = cv2.GaussianBlur(confidence_map, (101, 101), 0)
	
	binary_mask = (confidence_map > fog_threshold).astype(np.uint8) * 255
	morph_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
	binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, morph_kernel)
	binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, morph_kernel)
	
	return binary_mask, confidence_map

## Stage 1：Global Warping
計算每一幀到全域畫布（Global Canvas）的變換矩陣 `H`。

1. **Naive Focus Mask**：忽略畫面上下各 25%（上方為天空與雲、下方為水流），只在中央剛體區域取特徵
2. **SIFT**：每幀最多擷取 2000 個特徵點
3. **FLANN**：以 KD-tree 做 kNN 配對（k=2），再以 Lowe ratio test（0.75）篩選好的配對
4. **4-DOF Affine**（`compute_affine_matrix`）：以 `estimateAffinePartial2D` + RANSAC 只估計平移、旋轉、縮放，維持形狀以確保中位數融合後的邊緣銳利；縮放比例鉗制在 [0.98, 1.02]，避免特徵誤配造成的「呼吸效應」
5. **Keyframe Anchoring**（`align_video_keyframe_anchored`）：當前幀與固定錨點幀比對，而非只和前一幀比；當配對數 < 30 或位移超過畫面 15% 時重設錨點
6. **Gaussian Trajectory Smoothing**（`smooth_trajectory`）：將 `H` 拆成 X/Y 平移、旋轉角、縮放四個物理量，各自做 1D 高斯濾波後重組，消除果凍效應
7. **Middle Plane Transfer**：將座標原點移到中間幀，平均分散投影變形；最後計算畫布大小並加上平移偏移

In [3]:
def compute_affine_matrix(kp_curr, kp_ref, good_matches):
	"""
	計算變換矩陣：堅守 4 自由度 (剛體約束)
	這能維持影像的幾何形狀 (Shape Preserving)，確保 Median 融合後的邊緣銳利度。
	"""
	if len(good_matches) < 15: return None
	src_pts = np.float32([kp_curr[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
	dst_pts = np.float32([kp_ref[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
	
	# 實作 CVPR 2022 精神：強制使用相似變換 (4 DOF)，排除會破壞結構的錯切 (Shear)
	affine_matrix, _ = cv2.estimateAffinePartial2D(src_pts, dst_pts, method=cv2.RANSAC, ransacReprojThreshold=2.0)
	
	if affine_matrix is None:
		return None
	
	# 單獨提取縮放比例並施加限制，避免畫面異常放大縮小
	scale = np.sqrt(affine_matrix[0, 0]**2 + affine_matrix[1, 0]**2)
	clamped_scale = np.clip(scale, 0.98, 1.02)
	if scale > 0:
		affine_matrix[0:2, 0:2] = (affine_matrix[0:2, 0:2] / scale) * clamped_scale
	
	H_temp = np.eye(3, dtype=np.float32)
	H_temp[0:2, :] = affine_matrix
	return H_temp


def smooth_trajectory(H_list, window_size=11):
	"""
	軌跡平滑：使用高斯濾波器 (Gaussian Filter) 取代傳統移動平均
	消除高頻抖動的同時，不會產生「橡皮筋拖拽感」(果凍效應)。
	"""
	if len(H_list) < window_size:
		return H_list
		
	# 將矩陣解構為物理量：[X平移, Y平移, 旋轉角度, 縮放比例]
	params = np.array([[H[0, 2], H[1, 2], np.arctan2(H[1, 0], H[0, 0]), np.sqrt(H[0, 0]**2 + H[1, 0]**2)] for H in H_list])
	smoothed_params = np.copy(params)
	
	# 生成高斯核心
	sigma = window_size / 3.0
	x = np.arange(-window_size // 2 + 1., window_size // 2 + 1.)
	kernel = np.exp(-x**2 / (2 * sigma**2))
	kernel = kernel / np.sum(kernel)
	pad_size = window_size // 2
	
	# 對四個物理量分別進行高斯平滑
	for i in range(4):
		padded = np.pad(params[:, i], (pad_size, pad_size), mode='reflect')
		smoothed_params[:, i] = np.convolve(padded, kernel, mode='valid')
		
	# 將平滑後的物理量重組回齊次矩陣
	smoothed_H_list = []
	for i in range(len(H_list)):
		tx, ty, theta, scale = smoothed_params[i]
		H = np.eye(3, dtype=np.float32)
		H[0, 0] = scale * np.cos(theta)
		H[0, 1] = -scale * np.sin(theta)
		H[1, 0] = scale * np.sin(theta)
		H[1, 1] = scale * np.cos(theta)
		H[0, 2] = tx
		H[1, 2] = ty
		smoothed_H_list.append(H)
		
	return smoothed_H_list


def align_video_keyframe_anchored(video_path, total_frames):
	"""
	全域對齊主程式：結合物件級別遮罩 (OBJ-GSP) 與 中間平面錨定 (Middle Plane)
	"""
	cap = cv2.VideoCapture(video_path)
	ret, frame_curr = cap.read()
	h, w = frame_curr.shape[:2]
	
	sift = cv2.SIFT_create(nfeatures=2000)
	FLANN_INDEX_KDTREE = 1
	index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
	search_params = dict(checks=50)
	matcher = cv2.FlannBasedMatcher(index_params, search_params)

	# AAAI 2025 (OBJ-GSP) 精神：專注於中央剛體物件
	# 建立遮罩忽略頂部 25% (濃霧/天空) 與 底部 25% (流動河流)
	focus_mask = np.zeros((h, w), dtype=np.uint8)
	focus_mask[int(h*0.25):int(h*0.75), :] = 255

	gray_curr = cv2.cvtColor(frame_curr, cv2.COLOR_BGR2GRAY)
	anchor_kp, anchor_des = sift.detectAndCompute(gray_curr, mask=focus_mask)
	H_anchor_to_global = np.eye(3, dtype=np.float32)
	
	prev_kp, prev_des = anchor_kp, anchor_des
	raw_H_list = [np.eye(3, dtype=np.float32)]
	
	print("[Phase 1] Alignment with dynamic anchoring...")
	for i in tqdm(range(1, total_frames)):
		ret, frame_curr = cap.read()
		if not ret: break
		
		gray_curr = cv2.cvtColor(frame_curr, cv2.COLOR_BGR2GRAY)
		# 每一幀都套用遮罩，防止 SIFT 抓到水流等非剛體物件
		curr_kp, curr_des = sift.detectAndCompute(gray_curr, mask=focus_mask)
		
		if curr_des is None or anchor_des is None:
			H_curr_to_anchor = None
			good_matches = []
		else:
			matches = matcher.knnMatch(curr_des, anchor_des, k=2)
			good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
			H_curr_to_anchor = compute_affine_matrix(curr_kp, anchor_kp, good_matches)
		
		drift_x = abs(H_curr_to_anchor[0, 2]) if H_curr_to_anchor is not None else 0
		drift_y = abs(H_curr_to_anchor[1, 2]) if H_curr_to_anchor is not None else 0
		
		# 視角偏移過大，更換錨點
		if (
			H_curr_to_anchor is None
			or len(good_matches) < 30
			or drift_x > w * 0.15
			or drift_y > h * 0.15
		):
			anchor_kp, anchor_des = prev_kp, prev_des
			H_anchor_to_global = raw_H_list[-1]
			
			if curr_des is not None and anchor_des is not None:
				matches = matcher.knnMatch(curr_des, anchor_des, k=2)
				good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
				H_curr_to_anchor = compute_affine_matrix(curr_kp, anchor_kp, good_matches)
			else:
				H_curr_to_anchor = None
		
		if H_curr_to_anchor is not None:
			H_curr_to_global = H_anchor_to_global @ H_curr_to_anchor
		else:
			H_curr_to_global = raw_H_list[-1] 
		
		raw_H_list.append(H_curr_to_global)
		prev_kp, prev_des = curr_kp, curr_des
	
	cap.release()
	
	print("[Phase 1] Applying Gaussian trajectory smoothing...")
	raw_H_list = smooth_trajectory(raw_H_list, window_size=11)
	
	# =========================================================================
	# 實作 ICCV 2025 (PixelStitch) 概念：Middle Plane 轉移
	# =========================================================================
	print("[Phase 1] Performing Middle Plane Transformation...")
	mid_index = len(raw_H_list) // 2
	ret, H_mid_inv = cv2.invert(raw_H_list[mid_index])
	if ret:
		# 將所有矩陣乘上「中間幀的反矩陣」，等同於把世界座標的原點定在影片正中央
		raw_H_list = [H_mid_inv @ H for H in raw_H_list]
	# =========================================================================
	
	# Calculate global panorama size
	corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
	all_corners = np.concatenate([cv2.perspectiveTransform(corners, H) for H in raw_H_list], axis=0)
	
	[x_min, y_min] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
	[x_max, y_max] = np.int32(all_corners.max(axis=0).ravel() + 0.5)
	pano_w, pano_h = x_max - x_min, y_max - y_min
	
	# T_offset 是為了不讓影像超出畫布左上角邊界
	T_offset = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float32)
	final_H_list = [T_offset @ H for H in raw_H_list]
	
	print(f"[Phase 1] Global Canvas size: {pano_w}x{pano_h}")
	return final_H_list, pano_w, pano_h

## Stage 2：Temporal Fusion
`construct_global_image` 將所有幀融合成一張無霧的全域畫布。

1. 每一幀計算 Fog Confidence Map，連同影像一起以 `H` warp 到全域畫布（影像用 Lanczos 插值），並將有效範圍內縮以避開邊緣殘影
2. 在 GPU 上以 512×512 的 tile 分塊，對每個像素的時間序列做**自適應篩選**：
   - a = 該像素 confidence 的第 5 百分位（最不像霧），b = 第 95 百分位（最像霧）
   - 只保留 confidence ≤ `a + (b - a) × 0.25` 的樣本
   - 若該像素的平均亮度 > 220（本來就偏白的區域），則保留全部樣本
3. 對篩選後的樣本取**逐像素中位數**，最後以 unsharp mask 銳化

若傳入 `debug_video_path`，會另外輸出每一幀對齊結果（綠框為有效範圍）的 debug 影片。

In [4]:
import cv2
import numpy as np
import cupy as cp
import gc
import os
from tqdm import tqdm

def construct_global_image(video_path, H_list, pano_w, pano_h, total_frames, debug_video_path=None):
	"""
	建構全域畫布，並可選性地輸出每幀對齊狀態的 Debug 影片。
	"""
	cap = cv2.VideoCapture(video_path)
	sample_indices = list(range(0, total_frames))
	num_samples = len(sample_indices)
	
	print(f"\n[Phase 2] Constructing global canvas in RAM...")
	
	# 新增與修正：初始化 Debug 影片寫入器，並計算安全、標準化的影片尺寸
	debug_writer = None
	debug_w, debug_h = pano_w, pano_h
	
	if debug_video_path:
		print(f"[DEBUG] Alignment debug video will be saved to: {debug_video_path}")
		debug_writer = cv2.VideoWriter(debug_video_path, cv2.VideoWriter_fourcc(*'avc1'), 30, (debug_w, debug_h))
	
	canvas_samples = np.full((num_samples, pano_h, pano_w, 3), np.nan, dtype=np.float32)
	canvas_confs = np.full((num_samples, pano_h, pano_w), np.nan, dtype=np.float32) 
	
	sample_idx = 0
	for t in tqdm(range(total_frames), desc="Mapping frames to global canvas"):
		ret, frame = cap.read()
		if not ret: break
			
		is_sampled = t in sample_indices
		if not is_sampled and debug_writer is None:
			continue
			
		H = H_list[t]
		
		_, conf_map = get_confidence_map(frame)
		
		warped_frame = cv2.warpPerspective(frame, H, (pano_w, pano_h), flags=cv2.INTER_LANCZOS4).astype(np.float32)
		warped_conf = cv2.warpPerspective(conf_map, H, (pano_w, pano_h), flags=cv2.INTER_LINEAR)
		
		frame_h, frame_w = frame.shape[:2]
		footprint = np.ones((frame_h, frame_w), dtype=np.uint8) 
		warped_footprint = cv2.warpPerspective(footprint, H, (pano_w, pano_h))
		warped_footprint = cv2.erode(warped_footprint, np.ones((5, 5), np.uint8), iterations=3)
		valid_mask_2d = warped_footprint > 0
		
		# =========================================================
		# 寫入 Debug 影片前進行畫面縮放
		# =========================================================
		if debug_writer is not None:
			debug_vis = np.clip(warped_frame, 0, 255).astype(np.uint8)
			contours, _ = cv2.findContours(warped_footprint, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
			cv2.drawContours(debug_vis, contours, -1, (0, 255, 0), 3)
			
			cv2.putText(debug_vis, f"Global Alignment | Frame: {t}", (50, 80), 
						cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)
				
			debug_writer.write(debug_vis)
		# =========================================================
		
		if is_sampled:
			canvas_samples[sample_idx][valid_mask_2d] = warped_frame[valid_mask_2d]
			canvas_confs[sample_idx][valid_mask_2d] = warped_conf[valid_mask_2d]
			sample_idx += 1

	if debug_writer:
		debug_writer.release()

	print("[Phase 2] Self-adaptive median filtering...")
	global_canvas_bg = np.zeros((pano_h, pano_w, 3), dtype=np.uint8)
	
	tile_size = 512
	total_tiles = ((pano_h + tile_size - 1) // tile_size) * ((pano_w + tile_size - 1) // tile_size)
	
	stream = cp.cuda.Stream(non_blocking=True)
	
	with tqdm(total=total_tiles, desc="GPU Chunked") as pbar:
		for y in range(0, pano_h, tile_size):
			y_end = min(y + tile_size, pano_h)

			for x in range(0, pano_w, tile_size):
				x_end = min(x + tile_size, pano_w)
				
				tile_samples_cpu = canvas_samples[:, y:y_end, x:x_end, :]
				tile_confs_cpu = canvas_confs[:, y:y_end, x:x_end]
				
				if np.all(np.isnan(tile_confs_cpu)):
					pbar.update(1)
					continue
				
				with stream:
					gpu_samples = cp.asarray(tile_samples_cpu)
					gpu_confs = cp.asarray(tile_confs_cpu)
					
					gpu_confs_filled = cp.where(cp.isnan(gpu_confs), cp.inf, gpu_confs)
					sorted_confs = cp.sort(gpu_confs_filled, axis=0)
					
					valid_counts = cp.sum(~cp.isnan(gpu_confs), axis=0)
					
					idx_5 = cp.clip(cp.floor(valid_counts * 0.05).astype(cp.int32), 0, cp.maximum(valid_counts - 1, 0))
					conf_min = cp.take_along_axis(sorted_confs, idx_5[None, ...], axis=0)
					
					idx_95 = cp.clip(cp.floor(valid_counts * 0.95).astype(cp.int32), 0, cp.maximum(valid_counts - 1, 0))
					conf_max = cp.take_along_axis(sorted_confs, idx_95[None, ...], axis=0)
					
					adaptive_threshold = conf_min + (conf_max - conf_min) * 0.25
					adaptive_mask = gpu_confs <= adaptive_threshold
					
					avg_brightness = cp.nanmean(cp.nanmean(gpu_samples, axis=0), axis=-1)
					override_mask = avg_brightness > 220
					
					final_mask = adaptive_mask | override_mask
					filtered_samples = cp.where(final_mask[..., None], gpu_samples, cp.nan)
					
					tile_bg_gpu = cp.nanmedian(filtered_samples, axis=0)
					tile_bg_gpu = cp.nan_to_num(tile_bg_gpu, nan=0.0).astype(cp.uint8)
					
					tile_bg_cpu = cp.asnumpy(tile_bg_gpu)
				
				stream.synchronize()
				global_canvas_bg[y:y_end, x:x_end] = tile_bg_cpu
				
				del gpu_samples, gpu_confs, gpu_confs_filled, sorted_confs
				del conf_min, conf_max, adaptive_threshold, adaptive_mask
				del avg_brightness, override_mask, final_mask, filtered_samples, tile_bg_gpu
				cp.get_default_memory_pool().free_all_blocks()
				
				pbar.update(1)

	del canvas_samples, canvas_confs
	gc.collect()

	blurred_bg = cv2.GaussianBlur(global_canvas_bg, (0, 0), 2.0)
	global_canvas_bg = cv2.addWeighted(global_canvas_bg, 1.5, blurred_bg, -0.5, 0)

	return global_canvas_bg


c:\miniconda3\envs\DIP\lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


## Stage 3：Color Matching & Reprojection
`construct_defog_vedio` 將全域畫布反投影回每一幀的視角，並與原始幀融合。

1. 以 `H` 的反矩陣將全域畫布 warp 回當前視角
2. **Reinhard 色彩匹配（Lab 空間）**：
   - `calculate_lab_stats`：在 confidence < 0.4（較無霧）的區域計算 Lab 各通道的平均與標準差；有效像素少於畫面 3% 時跳過該幀
   - 以 **EMA（α = 0.1）** 平滑統計量，避免曝光逐幀跳動
   - `apply_exposure_alignment`：依統計量調整背景，縮放比例鉗制在 [0.5, 2]、L 通道位移鉗制在 ±60
3. 以霧遮罩 × 有效背景範圍（經羽化）作為權重，融合原始幀與對齊後的背景

為了加速，遮罩與統計量都在 1/4 解析度下計算。

In [5]:
import cv2
import numpy as np
from tqdm import tqdm

def calculate_lab_stats(img_bgr, valid_mask):
	"""計算 LAB 空間的均值與標準差 (支援低解析度計算)"""
	
	# 終極防線 1：拒絕「少數極端像素」！
	# 100 顆像素太少，如果剛好都在陰影裡會引發標準差塌陷。
	# 這裡要求至少要有畫面 3% 的有效樣本，否則寧可這幀不更新 EMA
	min_pixels = (img_bgr.shape[0] * img_bgr.shape[1]) * 0.03
	if np.sum(valid_mask) < max(1000, min_pixels):
		return None
	
	lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
	stats = []
	for i in range(3):
		c = lab[:, :, i]
		mu = np.mean(c[valid_mask])
		std = np.std(c[valid_mask]) + 1e-3 # 加上 1e-3 避免絕對的除以 0
		stats.append([mu, std])
	
	return np.array(stats)

def apply_exposure_alignment(bg_bgr, bg_stats, raw_stats):
	"""套用平滑後的統計數據進行曝光對齊 (NumPy 向量化加速)"""
	bg_lab = cv2.cvtColor(bg_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
	
	mu_bg = bg_stats[:, 0].reshape(1, 1, 3)
	std_bg = bg_stats[:, 1].reshape(1, 1, 3)
	mu_raw = raw_stats[:, 0].reshape(1, 1, 3)
	std_raw = raw_stats[:, 1].reshape(1, 1, 3)
	
	# 終極防線 2：數學鉗制 (Clamping)！阻止畫面變黑洞
	scale = std_raw / std_bg
	# 限制對比度縮放：在影片處理中，自然曝光極少會超過 0.5 到 2.0 倍的極端拉伸
	scale = np.clip(scale, 0.5, 2.0) 
	
	shift = mu_raw - mu_bg * scale
	# 限制 L 通道 (亮度, i=0) 的飄移量，不允許瞬間暴跌或暴增
	shift[0, 0, 0] = np.clip(shift[0, 0, 0], -60, 60)
	
	aligned_lab = bg_lab * scale + shift
	aligned_bg = np.clip(aligned_lab, 0, 255).astype(np.uint8)
	
	return cv2.cvtColor(aligned_bg, cv2.COLOR_LAB2BGR)

def construct_defog_vedio(video_path, H_list, total_frames, global_image):
	cap = cv2.VideoCapture(video_path)
	print("\n[Phase 3] Color matching & re-projection...")
	final_dehazed_frames = []
	cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
	
	ema_bg_stats = None
	ema_raw_stats = None
	alpha = 0.1 
	
	for t in tqdm(range(total_frames), desc="Rendering"):
		ret, frame_raw = cap.read()
		if not ret: break
			
		raw_h, raw_w = frame_raw.shape[:2]
		ret_inv, H_inv = cv2.invert(H_list[t])
		if not ret_inv: H_inv = np.eye(3)
		
		# 依然使用極速的雙線性插值
		bg_at_current_view = cv2.warpPerspective(global_image, H_inv, (raw_w, raw_h), flags=cv2.INTER_LINEAR)
		
		valid_bg_2d_strict = np.any(bg_at_current_view > 0, axis=-1).astype(np.uint8)
		
		# =================================================================
		# 終極加速 1：將原圖縮小至 1/4，再送進原本極度吃效能的 get_coarse_fog_mask
		# 這樣就連原函式裡面的 101x101 大模糊和形態學運算，也全部受惠於 16 倍的面積縮減！
		# =================================================================
		small_raw = cv2.resize(frame_raw, (raw_w // 4, raw_h // 4), interpolation=cv2.INTER_AREA)
		small_mask_coarse, small_conf_map = get_confidence_map(small_raw, fog_threshold=0)
		
		# 終極加速 2：在低解析度下直接完成高斯羽化，完成後再放大回原尺寸
		small_mask_float = small_mask_coarse.astype(np.float32) / 255.0
		small_blurred = cv2.GaussianBlur(small_mask_float, (21, 21), 0)
		mask_float = cv2.resize(small_blurred, (raw_w, raw_h), interpolation=cv2.INTER_LINEAR)
		
		# =================================================================
		# 終極加速 3：連 LAB 統計計算都在低解析度下執行 (捨棄全尺寸的轉換負擔)
		# =================================================================
		small_bg = cv2.resize(bg_at_current_view, (raw_w // 4, raw_h // 4), interpolation=cv2.INTER_AREA)
		valid_idx_small = small_conf_map < 0.4
		
		curr_bg_stats = calculate_lab_stats(small_bg, valid_idx_small)
		curr_raw_stats = calculate_lab_stats(small_raw, valid_idx_small)
		
		if curr_bg_stats is not None and curr_raw_stats is not None:
			if ema_bg_stats is None:
				ema_bg_stats = curr_bg_stats
				ema_raw_stats = curr_raw_stats
			else:
				ema_bg_stats = alpha * curr_bg_stats + (1.0 - alpha) * ema_bg_stats
				ema_raw_stats = alpha * curr_raw_stats + (1.0 - alpha) * ema_raw_stats
		
		# 套用對齊 (只有這裡是原圖運算，但已經完全向量化)
		if ema_bg_stats is not None and ema_raw_stats is not None:
			bg_at_current_view = apply_exposure_alignment(bg_at_current_view, ema_bg_stats, ema_raw_stats)
		
		# 處理背景有效邊界遮罩 (依然使用降維加速)
		mask_3c = np.stack([mask_float]*3, axis=-1)
		# valid_bg_2d = cv2.erode(valid_bg_2d_strict, np.ones((5, 5), np.uint8), iterations=3)
		
		valid_bg_float_raw = valid_bg_2d_strict.astype(np.float32)
		small_valid_bg = cv2.resize(valid_bg_float_raw, (raw_w // 4, raw_h // 4), interpolation=cv2.INTER_AREA)
		small_valid_bg_blurred = cv2.GaussianBlur(small_valid_bg, (21, 21), 0)
		valid_bg_float = cv2.resize(small_valid_bg_blurred, (raw_w, raw_h), interpolation=cv2.INTER_LINEAR)
		
		valid_bg_mask = np.expand_dims(valid_bg_float, axis=-1)
		final_mask = mask_3c * valid_bg_mask
		
		# 最終融合
		blended = frame_raw * (1.0 - final_mask) + bg_at_current_view * final_mask
		blended = np.clip(blended, 0, 255).astype(np.uint8)
		
		final_dehazed_frames.append(blended)
		
	cap.release()
	return final_dehazed_frames

## 主流程
`defog_video` 串接三個階段：

1. 讀取總幀數
2. Stage 1：`align_video_keyframe_anchored` 取得每幀的 `H` 與畫布大小
3. Stage 2：`construct_global_image` 產生全域畫布，存成 `output_image_path`
4. Stage 3：`construct_defog_vedio` 產生去霧後的每一幀，以 30 fps 輸出成 `output_video_path`

In [6]:
def defog_video(input_video_path, output_video_path, output_image_path, debug_video_path=None):
	if not os.path.exists(input_video_path):
		raise ValueError(f"Input video not found: {input_video_path}")

	cap = cv2.VideoCapture(input_video_path)
	total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
	cap.release()
	print(f"[DEBUG] Total frames: {total_frames}")

	# 1. 執行關鍵幀錨定對齊 (無區塊斷層、少累積扭曲)
	H_list, p_w, p_h = align_video_keyframe_anchored(input_video_path, total_frames)

	if not H_list:
		raise ValueError("Alignment failed: No valid homography matrices were computed.")

	dehazed_global_image = construct_global_image(input_video_path, H_list, p_w, p_h, total_frames, debug_video_path=debug_video_path)

	cv2.imwrite(output_image_path, dehazed_global_image)
	print(f"[Phase 2] Dehazed global image saved to: {output_image_path}")

	dehazed_video = construct_defog_vedio(
		input_video_path, H_list, total_frames, dehazed_global_image)

	frame_h, frame_w = dehazed_video[0].shape[:2]
	out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'avc1'), 30, (frame_w, frame_h))
	for f in dehazed_video: out.write(f)
	out.release()
	print(f"[Phase 3] Dehazed video saved to: {output_video_path}")


## 批次執行
處理 `./data/foggy_vid/1.mp4` ~ `4.mp4`，每支影片輸出去霧影片（`.mp4`）與全域畫布（`.jpg`）到 `./data/foggless_vid/`。

In [7]:
os.makedirs('./data/foggless_vid', exist_ok=True)
for i in range(1,5):
    output_name = f"./data/foggless_vid/{i}"
    defog_video(f'./data/foggy_vid/{i}.mp4', f'{output_name}.mp4', f'{output_name}.jpg')

[DEBUG] Total frames: 347
[Phase 1] Alignment with dynamic anchoring...


100%|██████████| 346/346 [00:19<00:00, 17.38it/s]


[Phase 1] Applying Gaussian trajectory smoothing...
[Phase 1] Performing Middle Plane Transformation...
[Phase 1] Global Canvas size: 1694x961

[Phase 2] Constructing global canvas in RAM...


Mapping frames to global canvas: 100%|██████████| 347/347 [00:30<00:00, 11.52it/s]


[Phase 2] Self-adaptive median filtering...


...

[Phase 2] Self-adaptive median filtering...


GPU Chunked: 100%|██████████| 6/6 [00:04<00:00,  1.42it/s]


[Phase 2] Dehazed global image saved to: ./data/foggless_vid/4.jpg

[Phase 3] Color matching & re-projection...


Rendering: 100%|██████████| 232/232 [00:13<00:00, 17.68it/s]


[Phase 3] Dehazed video saved to: ./data/foggless_vid/4.mp4
